# Data Preparation Pipeline
This notebook contains the complete pipeline for data preprocessing, data augmentation, and data distribution analysis.

## Step 1: Face Extraction (using MTCNN)
Extract student face crops from raw video files located in the `Video_/` directory using `facenet-pytorch` MTCNN detection. Crops are saved to `dataset/train_raw/`.

In [ ]:
import os
import cv2
import torch
import numpy as np
from facenet_pytorch import MTCNN
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

VIDEO_DIR = Path("Video_")
OUTPUT_DIR = Path("dataset/train_raw")
SAMPLE_RATE = 30
FACE_SIZE = 160
CONFIDENCE_THRESHOLD = 0.95

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

mtcnn = MTCNN(
    image_size=FACE_SIZE,
    margin=20,
    min_face_size=40,
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    post_process=True,
    device=device,
)


def parse_person_name(video_path: Path) -> str:
    stem = video_path.stem
    stem = stem.replace(" ", "_")
    return stem


def extract_faces_from_video(video_path: Path):
    person_name = parse_person_name(video_path)
    out_dir = OUTPUT_DIR / person_name

    sample_rates_to_try = [30, 15, 10, 5, 2, 1]
    saved_count = 0

    for sample_rate in sample_rates_to_try:
        # Clear old extracted faces to guarantee clean, noise-free starts
        if out_dir.exists():
            import shutil
            try:
                shutil.rmtree(out_dir)
            except Exception as e:
                logger.warning(f"Could not clear {out_dir}: {e}")
        out_dir.mkdir(parents=True, exist_ok=True)

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            logger.warning(f"  Cannot open {video_path}, skipping")
            return 0

        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = total_frames / fps if fps > 0 else 0

        frame_idx = 0
        saved_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_idx % sample_rate != 0:
                frame_idx += 1
                continue

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            boxes, probs = mtcnn.detect(rgb)

            if boxes is not None:
                # Filter and measure all detected faces
                valid_faces = []
                for i, box in enumerate(boxes):
                    if probs is not None and probs[i] < CONFIDENCE_THRESHOLD:
                        continue

                    box = box.astype(int)
                    x1, y1, x2, y2 = max(0, box[0]), max(0, box[1]), min(frame.shape[1], box[2]), min(frame.shape[0], box[3])
                    w, h = x2 - x1, y2 - y1
                    if w < 20 or h < 20:
                        continue
                    
                    # We save face area to select the largest one
                    area = w * h
                    valid_faces.append((area, x1, y1, x2, y2))

                if valid_faces:
                    # Select only the single largest face (the target student)
                    _, x1, y1, x2, y2 = max(valid_faces, key=lambda x: x[0])

                    face_crop = rgb[y1:y2, x1:x2]
                    face_resized = cv2.resize(face_crop, (FACE_SIZE, FACE_SIZE))
                    face_bgr = cv2.cvtColor(face_resized, cv2.COLOR_RGB2BGR)

                    fname = f"face_{saved_count:04d}.jpg"
                    cv2.imwrite(str(out_dir / fname), face_bgr)
                    saved_count += 1

            frame_idx += 1

        cap.release()

        # Stop retrying if we reached our target of at least 10 clean images
        if saved_count >= 10:
            logger.info(
                f"  {person_name}: successfully saved {saved_count} clean faces using sample_rate={sample_rate}"
            )
            break
        else:
            logger.info(
                f"  {person_name}: only obtained {saved_count} faces with sample_rate={sample_rate}. Retrying with higher frequency..."
            )
    return saved_count


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    video_extensions = {".mov", ".MOV", ".mp4", ".MP4", ".avi", ".AVI"}
    video_files = sorted(
        [f for f in VIDEO_DIR.iterdir() if f.suffix in video_extensions]
    )

    logger.info(f"Found {len(video_files)} videos in {VIDEO_DIR}")

    total_faces = 0
    for vf in video_files:
        n = extract_faces_from_video(vf)
        total_faces += n

    logger.info(f"Done! Extracted {total_faces} faces total from {len(video_files)} videos")

    logger.info("\nSummary per person:")
    for person_dir in sorted(OUTPUT_DIR.iterdir()):
        if person_dir.is_dir():
            count = len(list(person_dir.glob("*.jpg")))
            logger.info(f"  {person_dir.name}: {count} images")


if __name__ == "__main__":
    main()



## Step 2: Data Augmentation
Augment raw face crops in `dataset/train_raw/` using OpenCV transformations (horizontal flip, random rotation, random brightness/contrast, random Gaussian blur) to prevent overfitting. Augmented images are saved to `dataset/train_augmented/`.

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path
import random
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

INPUT_DIR = Path("dataset/train_raw")
OUTPUT_DIR = Path("dataset/train_augmented")
AUGMENTATIONS_PER_IMAGE = 4


def random_brightness_contrast(img):
    alpha = 1.0 + random.uniform(-0.2, 0.2)
    beta = random.randint(-30, 30)
    return cv2.convertScaleAbs(img, alpha=alpha, beta=beta)


def random_gaussian_blur(img):
    k = random.choice([3, 5])
    return cv2.GaussianBlur(img, (k, k), 0)


def random_rotation(img):
    h, w = img.shape[:2]
    angle = random.uniform(-10, 10)
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)


def augment_image(img):
    augs = {
        "flip": cv2.flip(img, 1),
        "brightness": random_brightness_contrast(img),
        "blur": random_gaussian_blur(img),
        "rotate": random_rotation(img),
    }
    return augs


def count_images_per_class(data_dir: Path) -> dict:
    counts = {}
    if not data_dir.exists():
        return counts
    for person_dir in sorted(data_dir.iterdir()):
        if person_dir.is_dir():
            n = len(list(person_dir.glob("*.jpg")))
            counts[person_dir.name] = n
    return counts


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    before_counts = count_images_per_class(INPUT_DIR)
    logger.info(f"Distribution BEFORE augmentation ({len(before_counts)} classes):")
    for name, count in sorted(before_counts.items()):
        logger.info(f"  {name}: {count}")

    for person_dir in sorted(INPUT_DIR.iterdir()):
        if not person_dir.is_dir():
            continue

        out_person = OUTPUT_DIR / person_dir.name
        out_person.mkdir(parents=True, exist_ok=True)

        images = sorted(person_dir.glob("*.jpg"))
        if not images:
            continue

        for img_path in images:
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            img_center = cv2.resize(img, (160, 160))
            cv2.imwrite(str(out_person / img_path.name), img_center)

            augs = augment_image(img_center)
            for aug_name, aug_img in augs.items():
                stem = img_path.stem
                aug_filename = f"{stem}_{aug_name}.jpg"
                cv2.imwrite(str(out_person / aug_filename), aug_img)

        n_before = len(images)
        n_after = len(list(out_person.glob("*.jpg")))
        logger.info(f"  {person_dir.name}: {n_before} -> {n_after} images")

    after_counts = count_images_per_class(OUTPUT_DIR)
    logger.info(f"\nDistribution AFTER augmentation ({len(after_counts)} classes):")
    for name, count in sorted(after_counts.items()):
        logger.info(f"  {name}: {count}")

    logger.info(f"\nAugmentation complete!")
    logger.info(f"  Before: {sum(before_counts.values())} total images")
    logger.info(f"  After:  {sum(after_counts.values())} total images")


if __name__ == "__main__":
    main()



## Step 3: Data Distribution & Embedding Analysis
Count raw and augmented images per class, plot distribution bar charts, and perform PCA (Principal Component Analysis) on cached FaceNet embeddings to visualize cluster separations in 2D.

In [ ]:
import numpy as np
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

RAW_DIR = Path("dataset/train_raw")
AUG_DIR = Path("dataset/train_augmented")
FIGURES_DIR = Path("figures")
EMBEDDING_FILE = Path("train_embeddings.npz")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "DejaVu Sans"],
    "font.size": 12,
    "axes.facecolor": "#f8f9fa",
    "figure.facecolor": "white",
})


def count_images(data_dir: Path) -> dict:
    counts = {}
    if not data_dir.exists():
        return counts
    for person_dir in sorted(data_dir.iterdir()):
        if person_dir.is_dir():
            n = len(list(person_dir.glob("*.jpg")))
            counts[person_dir.name] = n
    return counts


def plot_distribution(counts, title, filename, color="steelblue"):
    fig, ax = plt.subplots(figsize=(12, 5))
    names = list(counts.keys())
    values = list(counts.values())
    bars = ax.bar(range(len(names)), values, color=color, edgecolor="white",
                  linewidth=0.8, zorder=3)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Number of images", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight="bold", pad=12)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                str(val), ha="center", va="bottom", fontsize=7, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.3, zorder=0)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / filename.replace(".pdf", ".png"), bbox_inches="tight", dpi=200)
    plt.close(fig)
    logger.info(f"  Saved {FIGURES_DIR / filename.replace('.pdf', '.png')}")


def plot_embedding_pca(embeddings, labels, filename="embedding_pca.pdf"):
    if len(embeddings) < 3:
        logger.warning("Need >= 3 embeddings for PCA")
        return

    pca = PCA(n_components=2, random_state=42)
    emb_2d = pca.fit_transform(np.array(embeddings))

    fig, ax = plt.subplots(figsize=(12, 9))
    unique_labels = sorted(set(labels))
    palette = sns.color_palette("husl", len(unique_labels))
    label_color_map = dict(zip(unique_labels, palette))

    for lbl in unique_labels:
        mask = np.array([l == lbl for l in labels])
        ax.scatter(
            emb_2d[mask, 0], emb_2d[mask, 1],
            c=[label_color_map[lbl]], label=lbl.split("_")[0].title(),
            alpha=0.75, s=35, edgecolors="white", linewidth=0.4,
            zorder=3,
        )

    var1 = pca.explained_variance_ratio_[0] * 100
    var2 = pca.explained_variance_ratio_[1] * 100
    ax.set_xlabel(f"Principal Component 1 ({var1:.1f}%)", fontsize=12)
    ax.set_ylabel(f"Principal Component 2 ({var2:.1f}%)", fontsize=12)
    ax.set_title("PCA Projection of FaceNet Embeddings (512-d → 2-d)",
                 fontsize=15, fontweight="bold", pad=14)
    ax.legend(loc="best", fontsize=7, markerscale=0.8,
              frameon=True, facecolor="white", edgecolor="#ddd",
              ncol=2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.2, zorder=0)

    # inset: variance explained
    textstr = f"Total variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%"
    ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10,
            verticalalignment="top", bbox=dict(boxstyle="round,pad=0.4",
            facecolor="white", edgecolor="#ddd"))

    plt.tight_layout()
    fig.savefig(FIGURES_DIR / filename.replace(".pdf", ".png"), bbox_inches="tight", dpi=200)
    plt.close(fig)
    logger.info(f"  Saved {FIGURES_DIR / filename.replace('.pdf', '.png')}")


def main():
    logger.info("=== Data Distribution Analysis ===\n")

    before = count_images(RAW_DIR)
    after = count_images(AUG_DIR)

    logger.info(f"Before augmentation ({len(before)} classes): {sum(before.values())} total")
    for k, v in sorted(before.items()):
        logger.info(f"  {k}: {v}")

    logger.info("")
    logger.info(f"After augmentation ({len(after)} classes): {sum(after.values())} total")
    for k, v in sorted(after.items()):
        logger.info(f"  {k}: {v}")

    logger.info("\n--- Generating Figure 1: Distribution Before ---")
    plot_distribution(before, "Data Distribution Before Augmentation",
                      "distribution_before.pdf", color="#4361ee")

    logger.info("\n--- Generating Figure 2: Distribution After ---")
    plot_distribution(after, "Data Distribution After Augmentation",
                      "distribution_after.pdf", color="#f72585")

    logger.info("\n--- Generating Figure 3: Embedding PCA ---")
    if EMBEDDING_FILE.exists():
        data = np.load(EMBEDDING_FILE)
        embeddings = data["embeddings"]
        labels = data["labels"]
        logger.info(f"Loaded {len(embeddings)} embeddings from {EMBEDDING_FILE}")
        plot_embedding_pca(embeddings, labels)
    else:
        logger.warning(f"{EMBEDDING_FILE} not found. Skipping PCA plot.")

    logger.info("\nAll figures saved to figures/")


if __name__ == "__main__":
    main()

